# Расчетно-графическая работа: Идентификация диктора по голосу

**Дисциплина:** Технологии искусственного интеллекта и машинного обучения  
**Тема:** Распознавание и классификация звуковых сигналов  
**Вариант:** 50. Идентификация диктора (one-shot learning, triplet loss)

## 1. Введение

### Цель работы

Целью данной расчетно-графической работы является разработка системы верификации диктора по голосу (one-shot learning), которая определяет, принадлежит ли речь одному и тому же человеку, с использованием современных методов обработки аудиосигналов и нейронных сетей.

### Задачи работы

1. Изучить теоретические основы обработки звуковых сигналов и задач верификации диктора.
2. Подготовить аудиодатасет с записями речи различных дикторов (подмножество VoxCeleb / `test1`).
3. Выполнить предобработку аудио (ресемплинг, нормализация, преобразование в mel-спектрограммы).
4. Разработать архитектуру нейронной сети для построения эмбеддингов диктора и обучить её с использованием triplet loss.
5. Оценить качество верификации на тестовой выборке по метрикам (Accuracy, Precision, Recall, F1-score) и распределениям сходства для пар "один диктор" / "разные дикторы".
6. Реализовать демонстрационное программное приложение с графическим интерфейсом (Gradio) для проверки двух аудиозаписей.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torchaudio

from pathlib import Path

from speaker_verification.config import (
    DATA_ROOT,
    TARGET_SAMPLE_RATE,
    CLIP_DURATION_SECONDS,
)
from speaker_verification.audio_dataset import SpeakerAudioDataset
from speaker_verification.model import create_model

sns.set(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)

print("DATA_ROOT:", DATA_ROOT)
print("TARGET_SAMPLE_RATE:", TARGET_SAMPLE_RATE)
print("CLIP_DURATION_SECONDS:", CLIP_DURATION_SECONDS)

ModuleNotFoundError: No module named 'torch'

## 2. Описание задачи и постановка эксперимента

### 2.1. Описание задачи

Вариант 50 предполагает реализацию системы **верификации диктора** — определения, принадлежит ли пара аудиозаписей одному и тому же человеку. В отличие от классической многоклассовой классификации, здесь важна способность модели обобщать на **новых дикторов**, не представленных в обучении (one-shot / few-shot learning).

Ключевая идея:

- обучить сеть строить **эмбеддинги голоса** в таком пространстве, где
  - записи одного диктора находятся близко друг к другу,
  - записи разных дикторов — далеко друг от друга;
- сравнение двух записей сводится к вычислению сходства (косинусное расстояние) между их эмбеддингами.

### 2.2. Датасет `test1`

В работе используется датасет `test1` (подмножество VoxCeleb) со структурой:

```text
rgznemoe/speaker_verification/data/test1/
    speaker_01/
        0001.wav
        0002.wav
        ...
    speaker_02/
        0001.wav
        ...
    ...
```

Каждый подкаталог соответствует отдельному диктору.

**Основные характеристики датасета (заполняются после фактической подготовки):**

- количество дикторов: `N_speakers`;
- общее количество аудиозаписей: `N_files`;
- средняя длительность записей: ≈ `T` секунд;
- формат и частота дискретизации: WAV, исходно 16/44.1 кГц (приводится к 16 кГц).

### 2.3. Предобработка и представление аудио

Для обучения модели и последующей верификации выполняются следующие шаги:

1. **Ресемплинг** к частоте `16 кГц` (для единообразия и уменьшения объема данных).
2. **Приведение к фиксированной длине** (например, 3 секунды):
   - при необходимости запись обрезается или дополняется нулями.
3. **Преобразование к признаковому пространству**:
   - строятся **mel-спектрограммы** (`n_mels = 64`),
   - амплитуды переводятся в dB и нормализуются по частоте/времени.
4. (По желанию) Аугментации: добавление шума, изменение темпа/тона, временные сдвиги (могут быть включены для повышения устойчивости модели).

### 2.4. Архитектура и triplet loss

Для решения задачи используется эмбеддинговый подход с **triplet loss**:

- нейронная сеть принимает mel-спектрограмму и возвращает **вектор фиксированной размерности** (эмбеддинг диктора);
- во время обучения формируются тройки `(anchor, positive, negative)`:
  - `anchor` и `positive` — разные записи **одного и того же** диктора;
  - `negative` — запись **другого** диктора;
- triplet loss заставляет расстояние `d(anchor, positive)` быть меньше, чем `d(anchor, negative)` как минимум на значение `margin`.

Таким образом, задача верификации в продакшене сводится к сравнению двух эмбеддингов и проверке, превышает ли косинусное сходство заданный порог.

In [ ]:
# 2.5. Базовый анализ датасета: количество дикторов и записей

if DATA_ROOT.exists():
    dataset = SpeakerAudioDataset(DATA_ROOT)
    print("Всего аудиозаписей:", len(dataset))
    print("Количество дикторов:", len(dataset.speaker_to_idx))
    print("Список дикторов (speaker_id → индекс):")
    print(dataset.speaker_to_idx)
else:
    print("DATA_ROOT не найден. Заполните датасет test1 по структуре speaker_id/*.wav")

In [ ]:
# 2.6. Пример аудиосигнала и его mel-спектрограммы

if DATA_ROOT.exists():
    # Берем первый пример из датасета
    waveform, sp_idx = dataset[0]
    print(f"Пример 0: диктор #{sp_idx}, форма wave: {waveform.shape}")

    # Время в секундах для оси X
    t = np.linspace(0, CLIP_DURATION_SECONDS, waveform.shape[-1])

    fig, axs = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

    axs[0].plot(t, waveform.squeeze().numpy())
    axs[0].set_title("Временная форма сигнала")
    axs[0].set_ylabel("Амплитуда")

    # Mel-спектрограмма через тот же преобразователь, что и в модели
    from speaker_verification.model import MelSpecExtractor

    spec_extractor = MelSpecExtractor()
    with torch.no_grad():
        spec = spec_extractor(waveform.unsqueeze(0))  # (1, 1, n_mels, T_frames)
    spec = spec.squeeze(0).squeeze(0).cpu().numpy()

    im = axs[1].imshow(spec, origin="lower", aspect="auto", cmap="magma")
    axs[1].set_title("Log-mel спектрограмма")
    axs[1].set_xlabel("Время (фреймы)")
    axs[1].set_ylabel("Mel-частоты")
    fig.colorbar(im, ax=axs[1])

    plt.tight_layout()
    plt.show()
else:
    print("Пример не показан: DATA_ROOT отсутствует.")

In [ ]:
# 3.3. Просмотр архитектуры модели и числа параметров

model, device = create_model()
print(model)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Общее количество обучаемых параметров: {num_params:,}")

## 4. Процесс обучения модели

### 4.1. Конфигурация обучения и разделение данных

Обучающий код вынесен в модуль `speaker_verification.train` (скрипт для запуска из консоли), однако его логика может быть кратко описана следующим образом:

- используется весь датасет `test1` для обучения эмбеддингов (one-shot подход, задача — научиться общему признаковому пространству);
- батчи формируются через `DataLoader` по `SpeakerAudioDataset`;
- внутри батча по индексам дикторов генерируются тройки `(anchor, positive, negative)`;
- оптимизатор: `Adam(model.parameters(), lr=LEARNING_RATE)`;
- функция потерь: `TripletMarginLoss(margin=MARGIN, p=2)`;
- после обучения веса модели сохраняются в `speaker_verification/models/speaker_embedding_cnn.pt`.

Для запуска обучения (из консоли) используется команда:

```bash
python -m speaker_verification.train \
  --data_root speaker_verification/data/test1 \
  --model_path speaker_verification/models/speaker_embedding_cnn.pt \
  --batch_size 32 --epochs 20 --lr 1e-3 --margin 0.5
```

Ниже приведен пример кода, позволяющего запустить обучение из ноутбука (по желанию):

In [ ]:
# 4.2. (Опционально) Запуск обучения из ноутбука

from speaker_verification.train import train as train_model

# ВНИМАНИЕ: запуск обучения может занять значительное время.
# Раскомментируйте строку ниже для запуска (при наличии подготовленного датасета test1).

# train_model()

## 5. Интерпретация результатов: метрики и анализ сходства

После обучения модели важно оценить её способность различать голоса разных дикторов и объединять записи одного диктора.

Подход к оценке:

1. Случайным образом сформировать набор **пар аудиозаписей одного диктора** (genuine pairs).
2. Сформировать набор **пар разных дикторов** (impostor pairs).
3. Для всех пар вычислить эмбеддинги и косинусное сходство.
4. Построить распределения сходства для genuine / impostor пар и подобрать порог, максимизирующий целевую метрику (например, Accuracy или F1-score).
5. По выбранному порогу посчитать:
   - Accuracy;
   - Precision, Recall, F1-score;
   - (по желанию) ROC-кривую и AUC.

Ниже приведен пример кода для такого анализа.

In [ ]:
# 5.1. Оценка сходства для пар "один диктор" / "разные дикторы"

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

if DATA_ROOT.exists():
    # Загрузка обученной модели (если она сохранена)
    from speaker_verification.config import MODEL_PATH

    model_eval, device_eval = create_model()
    if MODEL_PATH.exists():
        state = torch.load(MODEL_PATH, map_location=device_eval)
        model_eval.load_state_dict(state)
        print(f"Модель загружена из {MODEL_PATH}")
    else:
        print("Внимание: обученная модель не найдена, используется случайно инициализированная модель.")

    model_eval.eval()

    # Готовим словарь: speaker_idx -> список индексов примеров
    speaker_to_indices = {}
    for idx, (_, sp_idx) in enumerate(dataset.items):
        speaker_to_indices.setdefault(sp_idx, []).append(idx)

    # Функция для получения эмбеддинга по индексу в датасете
    @torch.no_grad()
    def get_embedding(sample_idx: int) -> torch.Tensor:
        waveform, _ = dataset[sample_idx]
        waveform = waveform.unsqueeze(0).to(device_eval)  # (1, 1, T)
        emb = model_eval(waveform)
        return emb.squeeze(0).cpu()

    # Сэмплируем пары
    num_pairs = 100  # можно увеличить
    genuine_sims = []
    impostor_sims = []

    # Genuine: пары одного диктора
    rng = np.random.default_rng(42)
    for sp_idx, idxs in speaker_to_indices.items():
        if len(idxs) < 2:
            continue
        for _ in range(2):  # несколько пар на диктора
            i1, i2 = rng.choice(idxs, size=2, replace=False)
            e1 = get_embedding(i1)
            e2 = get_embedding(i2)
            sim = torch.nn.functional.cosine_similarity(e1, e2, dim=0).item()
            genuine_sims.append(sim)
            if len(genuine_sims) >= num_pairs:
                break
        if len(genuine_sims) >= num_pairs:
            break

    # Impostor: пары разных дикторов
    all_indices = np.arange(len(dataset.items))
    while len(impostor_sims) < num_pairs:
        i1, i2 = rng.choice(all_indices, size=2, replace=False)
        _, sp1 = dataset.items[i1]
        _, sp2 = dataset.items[i2]
        if sp1 == sp2:
            continue
        e1 = get_embedding(i1)
        e2 = get_embedding(i2)
        sim = torch.nn.functional.cosine_similarity(e1, e2, dim=0).item()
        impostor_sims.append(sim)

    # Визуализация распределений
    plt.figure(figsize=(8, 5))
    plt.hist(genuine_sims, bins=20, alpha=0.7, label="Один диктор")
    plt.hist(impostor_sims, bins=20, alpha=0.7, label="Разные дикторы")
    plt.xlabel("Косинусное сходство")
    plt.ylabel("Частота")
    plt.title("Распределение сходства эмбеддингов")
    plt.legend()
    plt.show()

    # Оценка метрик при фиксированном пороге
    threshold = 0.6  # можно подобрать по валидации
    y_true = [1] * len(genuine_sims) + [0] * len(impostor_sims)
    y_scores = genuine_sims + impostor_sims
    y_pred = [1 if s >= threshold else 0 for s in y_scores]

    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"Accuracy: {acc:.3f}")
    print(f"Precision: {prec:.3f}")
    print(f"Recall: {rec:.3f}")
    print(f"F1-score: {f1:.3f}")
else:
    print("Оценка не выполнена: DATA_ROOT отсутствует.")

## 6. Тестирование и демонстрация работы ПО

В рамках работы разработано простое демонстрационное приложение на базе библиотеки **Gradio** (`speaker_verification/app.py`). Приложение позволяет:

- загрузить две краткие аудиозаписи речи;
- получить оценку косинусного сходства эмбеддингов диктора;
- увидеть текстовый вывод о том, является ли это, по мнению модели, один и тот же диктор или разные.

Запуск приложения:

```bash
python -m speaker_verification.app
```

После запуска откроется веб-интерфейс (локальный адрес в консоли), в котором можно провести серию тестов и оценить поведение системы на реальных данных.

Кроме того, модуль `speaker_verification/verify.py` предоставляет CLI-инструмент для верификации двух файлов без графического интерфейса:

```bash
python -m speaker_verification.verify ref.wav test.wav
```

В пояснительной записке следует привести скриншоты интерфейса, примеры успешной и неуспешной верификации, а также обсудить ограничения и возможные улучшения (добавление аугментаций, усложнение архитектуры, использование предобученных эмбеддингов и т.п.).

## 7. Заключение

В работе реализована система верификации диктора по голосу с использованием triplet loss и сверточной эмбеддинг-сети поверх mel-спектрограмм. Показано, что даже относительно простая архитектура позволяет строить эмбеддинги, в которых записи одного диктора имеют высокое косинусное сходство, а записи разных дикторов — существенно ниже.

Проведена предобработка аудио, обучена модель на датасете `test1` (подмножество VoxCeleb), реализован набор метрик для оценки качества и создано демонстрационное приложение с графическим интерфейсом. На основе экспериментальных результатов можно сформулировать рекомендации по улучшению качества (увеличение объема данных, более сложные архитектуры, использование предобученных моделей, расширенный набор аугментаций и др.).

## 8. Приложения

- Полный исходный код модуля `speaker_verification` (датасеты, архитектура модели, обучение, верификация, GUI).
- Скрипты запуска обучения и приложения.
- Дополнительные графики и таблицы с результатами экспериментов.

## 3. Архитектура модели и конфигурация обучения

### 3.1. Архитектура эмбеддинг-сети

Для получения эмбеддингов диктора используется сверточная нейронная сеть `SpeakerEmbeddingCNN` (модуль `speaker_verification.model`), работающая поверх log-mel спектрограмм:

1. **Блок извлечения признаков** (`MelSpecExtractor`):
   - преобразует временной сигнал в мел-спектрограмму размерности `N_MELS × T_frames`;
   - применяет логарифмическое преобразование и нормализацию.

2. **Сверточная часть (CNN):**
   - несколько слоев `Conv2d + BatchNorm + ReLU + MaxPool/AdaptiveAvgPool` для сжатия спектрограммы в компактное представление;
   - итоговый тензор приводится к вектору фиксированной длины.

3. **Полносвязный слой:**
   - проецирует вектор в пространство размерности `EMBEDDING_DIM` (например, 128);
   - эмбеддинг нормируется по L2-норме (`nn.functional.normalize`).

Такое представление удобно для последующей работы с triplet loss и сравнения голосов через косинусное сходство.

### 3.2. Конфигурация обучения

Параметры обучения (см. `speaker_verification/config.py`):

- `BATCH_SIZE = 32` — размер батча;
- `NUM_EPOCHS = 20` — число эпох обучения;
- `LEARNING_RATE = 1e-3` — скорость обучения (оптимизатор Adam);
- `MARGIN = 0.5` — отступ (margin) для triplet loss;
- `TARGET_SAMPLE_RATE = 16000`, `CLIP_DURATION_SECONDS = 3.0` — параметры аудио;
- `N_MELS = 64`, `N_FFT = 1024`, `HOP_LENGTH = 256` — параметры мел-спектрограммы.

Функция потерь: **`TripletMarginLoss`** (PyTorch), оперирующая на батчах тройки `(anchor, positive, negative)`.